<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.3-first-gemini-call/notebooks/GCP_Capstone_1.3_First_Gemini_Call.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.3 Your First Gemini Call — Tokens, Cost & INR
**Netsetos GenAI Engineering — GCP Capstone**

Master the Gemini API: calls, token counting, cost calculation, streaming, and optimization.


## Cell 1: Setup


In [ ]:
!pip install -q google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
LOCATION = 'us-central1'
USD_TO_INR = 85


## Cell 2: Initialize Client & First Call


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What is a token in LLM terminology? Answer in 2 sentences.'
)
print('Response:', response.text)


## Cell 3: Decode usage_metadata


In [ ]:
meta = response.usage_metadata
print('Token Breakdown:')
print(f'  Input tokens:    {meta.prompt_token_count}')
print(f'  Output tokens:   {meta.candidates_token_count}')
print(f'  Thinking tokens: {getattr(meta, "thoughts_token_count", 0)}')
print(f'  Total tokens:    {meta.total_token_count}')


## Cell 4: Calculate Cost in INR


In [ ]:
PRICING = {
    'gemini-3.1-flash-lite': {'input': 0.25, 'output': 1.50},
    'gemini-3.6-flash':      {'input': 1.50, 'output': 7.50},
    'gemini-3.1-pro-preview':        {'input': 2.00, 'output': 12.00},
}

def calc_cost_inr(meta, model='gemini-3.6-flash'):
    p = PRICING[model]
    inp = meta.prompt_token_count * p['input'] / 1_000_000
    think = getattr(meta, 'thoughts_token_count', 0)
    out = (meta.candidates_token_count + think) * p['output'] / 1_000_000
    total = (inp + out) * USD_TO_INR
    print(f'  Input: Rs.{inp*USD_TO_INR:.4f} | Output+Think: Rs.{out*USD_TO_INR:.4f} | Total: Rs.{total:.4f}')
    print(f'  With $500: {500/(inp+out):,.0f} calls possible')
    return total

calc_cost_inr(response.usage_metadata)


## Cell 5: Free Token Counter


In [ ]:
texts = [
    'Hello',
    'Hyderabad is the capital of Telangana' * 10,
    'The transformer architecture uses self-attention' * 100,
]

for t in texts:
    r = client.models.count_tokens(model='gemini-3.6-flash', contents=t)
    print(f'  {r.total_tokens:>6,} tokens | {len(t):>6} chars | {len(t)/r.total_tokens:.1f} chars/token | {t[:40]}...')


## Cell 6: Compare 3 Models


In [ ]:
import time

prompt = 'Explain the difference between SQL and NoSQL. Give 2 examples of each.'
models = [
    ('gemini-3.1-flash-lite', 0.25, 1.50),
    ('gemini-3.6-flash', 1.50, 7.50),
    ('gemini-3.1-pro-preview', 2.00, 12.00),
]

print(f"{'Model':<35} {'Tokens':>7} {'Latency':>9} {'Rs':>9}")
print('-' * 65)
for name, ip, op in models:
    t0 = time.time()
    try:
        r = client.models.generate_content(model=name, contents=prompt)
        ms = (time.time()-t0)*1000
        u = r.usage_metadata
        cost = (u.prompt_token_count*ip + u.candidates_token_count*op)/1e6*85
        print(f'{name:<35} {u.total_token_count:>7} {ms:>7.0f}ms Rs.{cost:>6.4f}')
    except Exception as e:
        print(f'{name:<35} ERROR: {e}')


## Cell 7: Thinking Budget Impact


In [ ]:
prompt = 'What are the pros and cons of microservices?'

print(f"{'Budget':>8} {'Think':>7} {'Output':>8} {'Total':>7} {'Rs.':>9}")
for b in [0, 1024, 8192]:
    cfg = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=b)
    )
    r = client.models.generate_content(model='gemini-3.6-flash', contents=prompt, config=cfg)
    u = r.usage_metadata
    think = getattr(u, 'thoughts_token_count', 0)
    cost = (u.prompt_token_count*1.50 + (u.candidates_token_count+think)*7.50)/1e6*85
    print(f'{b:>8} {think:>7} {u.candidates_token_count:>8} {u.total_token_count:>7} Rs.{cost:>7.4f}')


## Cell 8: Streaming


In [ ]:
print('Streaming response:')
for chunk in client.models.generate_content_stream(
    model='gemini-3.6-flash',
    contents='Write a haiku about machine learning.'
):
    print(chunk.text, end='', flush=True)
print(f'\n\nTotal tokens: {chunk.usage_metadata.total_token_count}')


## Cell 9: Monthly Cost Projector


In [ ]:
def project_monthly(qpd, avg_in, avg_out, mix={'flash-lite':0.6,'flash':0.3,'pro':0.1}):
    prices = {'flash-lite':(0.25,1.50), 'flash':(1.50,7.50), 'pro':(2.00,12.00)}
    monthly = qpd * 30
    total = 0
    print(f'Queries/day: {qpd} | Monthly: {monthly:,}')
    for tier, pct in mix.items():
        ip, op = prices[tier]
        q = monthly * pct
        cost = (q*avg_in*ip + q*avg_out*op)/1e6
        total += cost
        print(f'  {tier:<12} {pct*100:>4.0f}% | {q:>7,.0f}q | ${cost:>6.2f} | Rs.{cost*85:>8.2f}')
    print(f'\n  TOTAL: ${total:.2f}/mo (Rs.{total*85:.2f})')
    print(f'  $500 lasts: {500/total:.1f} months ({500/total/12:.1f} years)')

project_monthly(100, 2000, 500)


## ✅ Module 1 Complete!

- ✅ 1.1: GCP project, billing, APIs, Cloud Shell
- ✅ 1.2: Dedicated SA, IAM roles, Secret Manager, ADC
- ✅ 1.3: Gemini API, tokens, INR cost, streaming, optimization

**Next: Module 2 — Tokens, Embeddings & Vector Search**
